# Dual-GPU DenseNet embedding on Kaggle

Attach three private Kaggle Datasets: project source, `data/reprocessed` content (`images/` and `metadata.jsonl`), and `best_E2_relation_aware_weighted_bce.pt`. This notebook creates raw 1024-D DenseNet embeddings and one FAISS index. It does not use RAD-DINO or Hugging Face.

In [ ]:
from pathlib import Path
import os
import shutil

# Change these three paths to your attached Kaggle Dataset slugs.
PROJECT_SOURCE_INPUT = Path('/kaggle/input/image-rag-project')
PROCESSED_INPUT = Path('/kaggle/input/iu-xray-1024px')
DENSENET_CHECKPOINT_INPUT = Path('/kaggle/input/image-rag-model/best_E2_relation_aware_weighted_bce.pt')
WORKING_DIR = Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'image-rag'
SHARD_DIR = WORKING_DIR / 'densenet-shards'
WORKER_DIR = WORKING_DIR / 'densenet-workers'
OUTPUT_DIR = WORKING_DIR / 'embeddings_densenet_raw'
BATCH_SIZE_PER_GPU = 32

assert PROJECT_SOURCE_INPUT.is_dir(), PROJECT_SOURCE_INPUT
assert (PROCESSED_INPUT / 'images').is_dir(), PROCESSED_INPUT / 'images'
assert (PROCESSED_INPUT / 'metadata.jsonl').is_file(), PROCESSED_INPUT / 'metadata.jsonl'
assert DENSENET_CHECKPOINT_INPUT.is_file(), DENSENET_CHECKPOINT_INPUT

shutil.rmtree(REPO_DIR, ignore_errors=True)
shutil.copytree(PROJECT_SOURCE_INPUT, REPO_DIR)
(REPO_DIR / 'model').mkdir(exist_ok=True)
shutil.copy2(DENSENET_CHECKPOINT_INPUT, REPO_DIR / 'model' / DENSENET_CHECKPOINT_INPUT.name)
os.chdir(REPO_DIR)

In [ ]:
import importlib.util
import subprocess
import sys

# Kaggle commonly includes these packages. Internet is only needed if this cell finds one missing.
required = {'faiss': 'faiss-cpu', 'pydicom': 'pydicom', 'yaml': 'pyyaml', 'dotenv': 'python-dotenv'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready')

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable the Kaggle GPU accelerator.'
assert torch.cuda.device_count() >= 2, f'Expected two GPUs, got {torch.cuda.device_count()}.'
for gpu_id in range(torch.cuda.device_count()):
    print(gpu_id, torch.cuda.get_device_name(gpu_id))

In [ ]:
import subprocess
import sys
from src.data.metadata import read_metadata_jsonl, write_metadata_jsonl

rows = list(read_metadata_jsonl(PROCESSED_INPUT / 'metadata.jsonl'))
assert rows and len({row['id'] for row in rows}) == len(rows)
shutil.rmtree(SHARD_DIR, ignore_errors=True)
SHARD_DIR.mkdir(parents=True)
shard_paths = []
for gpu_id in range(2):
    shard_path = SHARD_DIR / f'metadata-gpu{gpu_id}.jsonl'
    write_metadata_jsonl(rows[gpu_id::2], shard_path)
    shard_paths.append(shard_path)
    print(f'GPU {gpu_id}: {len(rows[gpu_id::2])} images')

In [ ]:
shutil.rmtree(WORKER_DIR, ignore_errors=True)
WORKER_DIR.mkdir(parents=True)
workers = []
checkpoint_path = REPO_DIR / 'model' / DENSENET_CHECKPOINT_INPUT.name

for gpu_id, shard_path in enumerate(shard_paths):
    worker_output = WORKER_DIR / f'gpu{gpu_id}'
    log_path = WORKER_DIR / f'gpu{gpu_id}.log'
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['PYTHONUNBUFFERED'] = '1'
    command = [
        sys.executable, 'scripts/compute_image_embeddings.py',
        '--processed_dir', str(PROCESSED_INPUT),
        '--metadata_path', str(shard_path),
        '--out_dir', str(worker_output),
        '--densenet_checkpoint', str(checkpoint_path),
        '--vector_db', 'none', '--device', 'cuda',
        '--batch_size', str(BATCH_SIZE_PER_GPU), '--clean',
    ]
    log_handle = log_path.open('w', encoding='utf-8')
    process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=log_handle, stderr=subprocess.STDOUT)
    workers.append((gpu_id, process, log_handle, log_path, worker_output))

for gpu_id, process, log_handle, log_path, _ in workers:
    exit_code = process.wait()
    log_handle.close()
    if exit_code:
        print(log_path.read_text(encoding='utf-8'))
        raise RuntimeError(f'GPU {gpu_id} worker failed with exit code {exit_code}.')
    print(f'GPU {gpu_id} complete: {log_path}')

In [ ]:
import faiss
import json
import numpy as np

vectors_by_id = {}
manifest = None
for _, _, _, _, worker_output in workers:
    vectors = np.load(worker_output / 'embeddings.npy')
    worker_ids = (worker_output / 'ids.txt').read_text(encoding='utf-8').splitlines()
    assert len(vectors) == len(worker_ids)
    assert not (set(worker_ids) & set(vectors_by_id))
    vectors_by_id.update(zip(worker_ids, vectors))
    if manifest is None:
        manifest = json.loads((worker_output / 'embedding_manifest.json').read_text(encoding='utf-8'))

ordered_ids = [row['id'] for row in rows]
assert set(ordered_ids) == set(vectors_by_id)
embeddings = np.ascontiguousarray(np.vstack([vectors_by_id[item_id] for item_id in ordered_ids]), dtype=np.float32)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.mkdir(parents=True)
np.save(OUTPUT_DIR / 'embeddings.npy', embeddings)
(OUTPUT_DIR / 'ids.txt').write_text('\n'.join(ordered_ids) + '\n', encoding='utf-8')
write_metadata_jsonl(rows, OUTPUT_DIR / 'metadata.jsonl')
(OUTPUT_DIR / 'embedding_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(OUTPUT_DIR / 'faiss.index'))
print(f'Saved {len(embeddings)} DenseNet embeddings with dimension {embeddings.shape[1]}.')

In [ ]:
archive_path = shutil.make_archive(str(WORKING_DIR / 'embeddings_densenet_raw'), 'zip', root_dir=OUTPUT_DIR)
print(f'Download this Kaggle output: {archive_path}')